# 02 · Pipeline medalhão em execução

Este notebook **executa a pipeline de verdade** — não descreve. Ao final, o lakehouse
local está populado e o relatório de qualidade, gerado.

Ordem: `raw` → **bronze** → **silver** → **gold**, com o contrato de dados aplicado
em cada fronteira.

In [1]:
import sys
from pathlib import Path

# Caminho relativo: o notebook encontra o projeto a partir da própria posição,
# então funciona em qualquer máquina, sem editar nada.
RAIZ = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(RAIZ))

import pandas as pd
import matplotlib.pyplot as plt

pd.set_option("display.max_columns", 60)
pd.set_option("display.width", 160)
plt.rcParams.update({
    "figure.figsize": (10, 4.5),
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.grid": True,
    "grid.alpha": 0.25,
    "font.size": 10,
})
print("Raiz do projeto:", RAIZ.name)

Raiz do projeto: tc2


In [2]:
import os
# Parquet no notebook evita depender de download de jar do Delta na primeira execução.
# Em Docker e no Databricks, FORMATO_TABELA=delta é o padrão.
os.environ.setdefault("FORMATO_TABELA", "parquet")

from src.logging_conf import configurar
from src import spark_session
from src.config import CFG

configurar("INFO")
spark = spark_session.criar()
print("Formato efetivo:", spark_session.FORMATO_EFETIVO)

Picked up JAVA_TOOL_OPTIONS: -Djavax.net.ssl.trustStore=/root/.ccr/java-truststore.p12 -Djavax.net.ssl.trustStorePassword=changeit -Djavax.net.ssl.trustStoreType=PKCS12 -Dhttps.proxyHost=127.0.0.1 -Dhttps.proxyPort=34139 -Dhttp.nonProxyHosts=localhost|127.0.0.1|::1|127.*|0.*|::|169.254.*|api.anthropic.com|api-staging.anthropic.com|api-pr-preview.anthropic.com|mcp-proxy.anthropic.com|mcp-proxy-staging.anthropic.com|registry.npmjs.org|jsr.io|npm.jsr.io|pypi.org|files.pythonhosted.org|index.crates.io|proxy.golang.org|host.docker.internal|10.*|172.16.*|172.17.*|172.18.*|172.19.*|172.20.*|172.21.*|172.22.*|172.23.*|172.24.*|172.25.*|172.26.*|172.27.*|172.28.*|172.29.*|172.30.*|172.31.*|192.168.*|100.64.0.0/10|*.svc.cluster.local|*.svc.cluster.local -Djdk.http.auth.tunneling.disabledSchemes= -Djdk.http.auth.proxying.disabledSchemes=


Picked up JAVA_TOOL_OPTIONS: -Djavax.net.ssl.trustStore=/root/.ccr/java-truststore.p12 -Djavax.net.ssl.trustStorePassword=changeit -Djavax.net.ssl.trustStoreType=PKCS12 -Dhttps.proxyHost=127.0.0.1 -Dhttps.proxyPort=34139 -Dhttp.nonProxyHosts=localhost|127.0.0.1|::1|127.*|0.*|::|169.254.*|api.anthropic.com|api-staging.anthropic.com|api-pr-preview.anthropic.com|mcp-proxy.anthropic.com|mcp-proxy-staging.anthropic.com|registry.npmjs.org|jsr.io|npm.jsr.io|pypi.org|files.pythonhosted.org|index.crates.io|proxy.golang.org|host.docker.internal|10.*|172.16.*|172.17.*|172.18.*|172.19.*|172.20.*|172.21.*|172.22.*|172.23.*|172.24.*|172.25.*|172.26.*|172.27.*|172.28.*|172.29.*|172.30.*|172.31.*|192.168.*|100.64.0.0/10|*.svc.cluster.local|*.svc.cluster.local -Djdk.http.auth.tunneling.disabledSchemes= -Djdk.http.auth.proxying.disabledSchemes=


Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/08/30 21:25:24 WARN Utils: Your hostname, vm, resolves to a loopback address: 127.0.0.1; using 192.0.2.2 instead (on interface eth0)
26/08/30 21:25:24 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address


Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).


26/08/30 21:25:24 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


2026-08-30T21:25:28 | INFO     | [SPARK] Sessão ativa | master=local[*] | formato=parquet | conector Kafka=não


Formato efetivo: parquet


## 1. Camada Bronze — o dado como chegou

Schema **explícito** (nunca `inferSchema`), metadados de linhagem em toda linha e
partição por data de ingestão para preservar o histórico.

In [3]:
from src.camadas import bronze

destinos = bronze.executar(spark)
list(destinos)

2026-08-30T21:25:28 | INFO     | ==============================================================================


2026-08-30T21:25:28 | INFO     | CAMADA BRONZE — ingestão batch | run_id=run-20260830T212528-4440cb


2026-08-30T21:25:28 | INFO     | ==============================================================================


2026-08-30T21:25:35 | INFO     | [DQ:BRONZE] PASS | uf | min_count | coluna=None | registros=27 minimo=27


2026-08-30T21:25:35 | INFO     | [DQ:BRONZE] PASS | uf | not_null | coluna=id_uf | 0 registro(s) (0.00%) violam not_null


...

2026-08-30T21:25:52 | INFO     | [DQ:BRONZE] PASS | contexto_socioeconomico_municipio | not_null | coluna=id_municipio_6dig | 0 registro(s) (0.00%) violam not_null


2026-08-30T21:25:52 | INFO     | [DQ:BRONZE] PASS | contexto_socioeconomico_municipio | range | coluna=idhm | 0 registro(s) (0.00%) violam range


2026-08-30T21:25:52 | INFO     | [DQ:BRONZE] contexto_socioeconomico_municipio | score=100.0% | validos=5564 | quarentena=0


2026-08-30T21:25:53 | INFO     | [OBS] bronze   | contexto_socioeconomico_municipio      | OK   | entrada=   5564 saida=   5564 quarentena=    0 |   2.30s


['uf',
 'municipio',
 'meta_alfabetizacao_brasil',
 'meta_alfabetizacao_uf',
 'meta_alfabetizacao_municipio',
 'aluno',
 'contexto_socioeconomico_municipio']

In [4]:
df_bronze = spark_session.ler_tabela(spark, CFG.camada("bronze/municipio"))
df_bronze.select("id_municipio", "nome_municipio", "sigla_uf",
                 "_ingestion_date", "_source_file", "_record_hash").show(5, truncate=40)

+------------+---------------------+--------+---------------+-------------+----------------------------------------+
|id_municipio|       nome_municipio|sigla_uf|_ingestion_date| _source_file|                            _record_hash|
+------------+---------------------+--------+---------------+-------------+----------------------------------------+
|     1100015|Alta Floresta D'Oeste|      RO|     2026-08-24|municipio.csv|bb9d90f98d9d3bf3d4aff4b44b32e7d59f4f2...|
|     1100023|            Ariquemes|      RO|     2026-08-24|municipio.csv|f5ac2cc3a441dffa6abcface84c3e2e706a04...|
|     1100031|               Cabixi|      RO|     2026-08-24|municipio.csv|e0e1d57a8e7de6321401b30610c2079220991...|
|     1100049|               Cacoal|      RO|     2026-08-24|municipio.csv|10c75bd6bc9adee9262a41e4b17afed9321e2...|
|     1100056|           Cerejeiras|      RO|     2026-08-24|municipio.csv|adcb0aa7856ee2ab9622dd938c2d1ef6c9c9e...|
+------------+---------------------+--------+---------------+---

## 2. Camada Silver — limpeza, padronização e integração

Aqui acontece o que o desafio pede: limpeza, tratamento de ausentes, padronização de
nomes e tipos, validação de consistência, normalização de chaves e **integração das
bases**.

In [5]:
from src.camadas import silver

tabelas = silver.executar(spark)
list(tabelas)

2026-08-30T21:25:53 | INFO     | ==============================================================================


2026-08-30T21:25:53 | INFO     | CAMADA SILVER — limpeza, padronização e integração das bases


2026-08-30T21:25:53 | INFO     | ==============================================================================


2026-08-30T21:25:59 | INFO     | [DQ:SILVER] PASS | dim_uf | min_count | coluna=None | registros=27 minimo=27


2026-08-30T21:25:59 | INFO     | [DQ:SILVER] PASS | dim_uf | unico | coluna=id_uf | 0 registro(s) duplicado(s)


...

2026-08-30T21:26:23 | INFO     | [DQ:SILVER] PASS | fato_aluno | range | coluna=idade | 0 registro(s) (0.00%) violam range


2026-08-30T21:26:23 | INFO     | [DQ:SILVER] PASS | fato_aluno | chave_estrangeira | coluna=id_municipio | 0 registro(s) (0.00%) chave(s) órfã(s) vs dim_municipio


2026-08-30T21:26:23 | INFO     | [DQ:SILVER] fato_aluno | score=100.0% | validos=59945 | quarentena=55


2026-08-30T21:26:25 | INFO     | [OBS] silver   | fato_aluno                             | OK   | entrada=  60000 saida=  59945 quarentena=   55 |   6.60s


['dim_uf',
 'dim_municipio',
 'fato_meta_brasil',
 'fato_indicador_uf',
 'fato_indicador_municipio',
 'fato_aluno']

In [6]:
# Antes x depois da padronização de texto
bruto = df_bronze.select("id_municipio", "nome_municipio").toPandas()
limpo = tabelas["dim_municipio"].select("id_municipio", "nome_municipio").toPandas()

comparacao = (bruto.rename(columns={"nome_municipio": "bronze"})
                   .merge(limpo.rename(columns={"nome_municipio": "silver"}), on="id_municipio"))
comparacao[comparacao["bronze"] != comparacao["silver"]].head(8)

,id_municipio,bronze,silver
0,1100015,Alta Floresta D'Oeste,Alta Floresta D'oeste
8,1100098,Espigão D'Oeste,Espigão D'oeste
12,1100130,Machadinho D'Oeste,Machadinho D'oeste
13,1100148,Nova Brasilândia D'Oeste,Nova Brasilândia D'oeste
20,1100296,Santa Luzia D'Oeste,Santa Luzia D'oeste
24,1100346,Alvorada D'Oeste,Alvorada D'oeste
44,1101484,São Felipe D'Oeste,São Felipe D'oeste
63,1200351,Marechal Thaumaturgo,Marechal Thaumaturgo


In [7]:
# A integração produziu contexto socioeconômico para quantos municípios?
dim = tabelas["dim_municipio"]
total = dim.count()
com_contexto = dim.filter("contexto_disponivel").count()
print(f"{com_contexto:,} de {total:,} municípios com contexto socioeconômico "
      f"({100 * com_contexto / total:.1f}%)")
print("Os demais foram criados depois do Censo 2010 — permanecem na base, marcados.")

5,564 de 5,571 municípios com contexto socioeconômico (99.9%)
Os demais foram criados depois do Censo 2010 — permanecem na base, marcados.


### Quarentena — para onde vai o que não passou

Registro reprovado não é descartado: vai para `_quarentena/` com o motivo na linha.

In [8]:
caminho_q = CFG.camada("_quarentena/silver/fato_aluno")
if spark_session.existe(caminho_q):
    q = spark_session.ler_tabela(spark, caminho_q)
    print(f"{q.count()} registro(s) em quarentena")
    q.select("id_aluno", "id_municipio", "proficiencia_saeb", "_motivo_quarentena").show(5, truncate=50)
else:
    print("Nenhum registro em quarentena nesta execução.")

55 registro(s) em quarentena
+----------------+------------+-----------------+-----------------------+
|        id_aluno|id_municipio|proficiencia_saeb|     _motivo_quarentena|
+----------------+------------+-----------------+-----------------------+
|12cc933706f08648|     2932903|             NULL|range:proficiencia_saeb|
|16229c9809fd35fb|     1504208|             NULL|range:proficiencia_saeb|
|3bd64c0d68b1b99a|     1200401|             NULL|range:proficiencia_saeb|
|4313aa2fbb3c9fcb|     3534401|             NULL|range:proficiencia_saeb|
|4ad7a4753bd43561|     2202778|             NULL|range:proficiencia_saeb|
+----------------+------------+-----------------+-----------------------+
only showing top 5 rows


## 3. Camada Gold — os produtos analíticos

In [9]:
from src.camadas import gold

produtos = gold.executar(spark)
for nome, df in produtos.items():
    print(f"{nome:38s} {df.count():>7,} registros | {len(df.columns):>3} colunas")

2026-08-30T21:26:27 | INFO     | ==============================================================================


2026-08-30T21:26:27 | INFO     | CAMADA GOLD — datasets analíticos


2026-08-30T21:26:27 | INFO     | ==============================================================================


2026-08-30T21:26:31 | INFO     | [DQ:GOLD] PASS | indicador_alfabetizacao_municipio | min_count | coluna=None | registros=16713 minimo=5000


2026-08-30T21:26:31 | INFO     | [DQ:GOLD] PASS | indicador_alfabetizacao_municipio | unico | coluna=ano,id_municipio | 0 registro(s) duplicado(s)


...

indicador_alfabetizacao_municipio       16,713 registros |  44 colunas


meta_vs_realizado_uf                        81 registros |  19 colunas


evolucao_temporal_brasil                     8 registros |  15 colunas


painel_desigualdade                         60 registros |  12 colunas


ranking_municipios                         600 registros |  16 colunas
features_ml_municipio                   16,713 registros |  27 colunas


### Reconciliação: o agregado bate com o publicado?

A tabela `meta_vs_realizado_uf` publica lado a lado o valor divulgado pelo INEP e o valor
recalculado a partir dos municípios. Se a divergência crescer um dia, ela aparece aqui —
e não numa reunião.

In [10]:
uf = produtos["meta_vs_realizado_uf"].filter("ano = 2024").toPandas()
print("Divergência máxima entre calculado e publicado:",
      f"{uf['divergencia_pp'].abs().max():.2f} p.p.")
uf[["sigla_uf", "indicador_publicado_pct", "indicador_calculado_pct",
    "divergencia_pp", "meta_pct", "gap_meta_pp", "status_meta"]].head(10)

Divergência máxima entre calculado e publicado: 0.00 p.p.


,sigla_uf,indicador_publicado_pct,indicador_calculado_pct,divergencia_pp,meta_pct,gap_meta_pp,status_meta
0,AC,51.4,51.4,0.0,52.7,-1.3,ABAIXO_DA_META
1,AL,48.6,48.6,0.0,50.3,-1.7,ABAIXO_DA_META
2,AM,49.2,49.2,0.0,56.0,-6.8,ABAIXO_DA_META
3,AP,46.6,46.6,0.0,48.5,-1.9,ABAIXO_DA_META
4,BA,36.0,36.0,0.0,43.1,-7.1,ABAIXO_DA_META
5,CE,85.3,85.3,0.0,84.3,1.0,ACIMA_DA_META
6,DF,59.1,59.1,0.0,59.3,-0.2,NA_META
7,ES,71.7,71.7,0.0,69.7,2.0,ACIMA_DA_META
8,GO,72.7,72.7,0.0,70.9,1.8,ACIMA_DA_META
9,MA,59.6,59.6,0.0,59.4,0.2,NA_META


## 4. Observabilidade da execução

In [11]:
from src.observabilidade import runs, relatorio

runs.persistir(spark)
caminho = relatorio.gerar()

eventos = pd.DataFrame([{
    "camada": e.camada, "tabela": e.tabela, "status": e.status,
    "entrada": e.registros_entrada, "saida": e.registros_saida,
    "quarentena": e.registros_quarentena, "qualidade": e.score_qualidade,
    "duracao_s": e.duracao_s,
} for e in runs.eventos()])
eventos

2026-08-30T21:26:47 | INFO     | [OBS] Execução run-20260830T212528-4440cb registrada em run-20260830T212528-4440cb.jsonl


2026-08-30T21:26:47 | INFO     | ------------------------------------------------------------------------------


2026-08-30T21:26:47 | INFO     | [OBS] Execução run-20260830T212528-4440cb | etapas=19 | falhas=0 | 71.9s | 20.60 MB | quarentena=55


2026-08-30T21:26:47 | INFO     | [OBS] Relatório: data/_observabilidade/relatorio.md


2026-08-30T21:26:47 | INFO     | ------------------------------------------------------------------------------


,camada,tabela,status,entrada,saida,quarentena,qualidade,duracao_s
0,bronze,uf,OK,27,27,0,100.0,8.155
1,bronze,municipio,OK,5593,5593,0,100.0,3.188
2,bronze,meta_alfabetizacao_brasil,OK,8,8,0,100.0,1.750
3,bronze,meta_alfabetizacao_uf,OK,216,216,0,100.0,1.657
4,bronze,meta_alfabetizacao_municipio,OK,16779,16779,0,100.0,2.519
5,bronze,aluno,OK,60240,60240,0,100.0,4.756
6,bronze,contexto_socioeconomico_municipio,OK,5564,5564,0,100.0,2.296
7,silver,dim_uf,OK,27,27,0,100.0,6.111
8,silver,dim_municipio,OK,5571,5571,0,100.0,7.572
9,silver,fato_meta_brasil,OK,8,8,0,100.0,2.424


In [12]:
# `encerrar` não para a sessão quando o notebook roda no Databricks — lá ela
# pertence à plataforma e pararia o notebook inteiro junto.
spark_session.encerrar(spark)
print("Pipeline concluída. Relatório em data/_observabilidade/relatorio.md")

Pipeline concluída. Relatório em data/_observabilidade/relatorio.md
